# SRGAN ×2 — 판별자 붕괴 전후 비교

입력은 실제 Sentinel-2 영상 그대로 두고, 목표만 **LR × 2** 크기로 잡았다.
학습 도중 판별자가 무너지는데, **무너지기 전과 후 중 어느 쪽이 더 좋은지** 직접 비교한다.

## 1. 데이터

In [ ]:
import urllib.request
LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'srgan_models.py', 'srgan_losses.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)

from sr_utils import *
SCALE = 2

val_lr, val_hr = pair('validation', REP['validation'])
val_tgt = retarget(val_hr, val_lr, SCALE)      # 목표를 LR x2 크기로
test_lr = load_test()

show([('validation (Paris)', val_lr, val_tgt), ('test (Incheon)', test_lr, None)])
print(f'입력 {val_lr.shape[0]}px  ->  목표 {val_tgt.shape[0]}px   (x{SCALE})')
print('입력 LR 은 실제 촬영본 그대로다. 목표 HR 만 줄였다.')

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 적은 데이터로 몇 번만 돌린다.
아래 3·4번은 전체 데이터로 100 epoch 학습해둔 가중치를 쓴다.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from srgan_models import Generator, Discriminator
from srgan_losses import GeneratorLoss

N_TRAIN, EPOCHS, BATCH = 16, 3, 4
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = [], []
for s in list_split('training')[:N_TRAIN]:
    l, h = pair('training', s)
    lo.append(l); hi.append(retarget(h, l, SCALE))
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
loader = DataLoader(TensorDataset(to_t(lo), to_t(hi)), batch_size=BATCH, shuffle=True)

netG, netD = Generator(SCALE).to(dev).train(), Discriminator().to(dev).train()
optG, optD = torch.optim.Adam(netG.parameters()), torch.optim.Adam(netD.parameters())
crit = GeneratorLoss().to(dev)

for ep in range(1, EPOCHS + 1):
    gl = dl = dx = dgz = 0.0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        fake = netG(x)                                     # 1) 생성자
        g_loss = crit(netD(fake).mean(), fake, y)
        optG.zero_grad(); g_loss.backward(); optG.step()

        real_out, fake_out = netD(y).mean(), netD(fake.detach()).mean()
        d_loss = 1 - real_out + fake_out                   # 2) 판별자
        optD.zero_grad(); d_loss.backward(); optD.step()

        gl += g_loss.item(); dl += d_loss.item()
        dx += real_out.item(); dgz += fake_out.item()
    k = len(loader)
    print(f'epoch {ep}/{EPOCHS}  Loss_G {gl/k:.4f}  Loss_D {dl/k:.4f}  '
          f'D(x) {dx/k:.3f}  D(G(z)) {dgz/k:.3f}')

## 3. 결과 비교 — 붕괴 전 vs 붕괴 후

전체 데이터로 100 epoch 돌린 기록이다. **epoch 43 에서 판별자가 무너졌다.**

In [ ]:
import pandas as pd

MODEL = f'{BASE}/models/03_srgan_x2'
e = pd.read_csv(fetch(f'{MODEL}/statistics/x2_train_results.csv', 'x2.csv'), index_col=0)
COLLAPSE = 43

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(e.index, e.Score_D, color='#2f6f9f', lw=1.6, label='D(x)  real')
ax[0].plot(e.index, e.Score_G, color='#c96a5b', lw=1.6, label='D(G(z))  fake')
ax[0].axvline(COLLAPSE, color='#333', ls='--', lw=1)
ax[0].set_title(f'Discriminator output (collapse at epoch {COLLAPSE})')
ax[0].set_ylim(-.05, 1.08); ax[0].legend(fontsize=8)

ax[1].plot(e.index, e.Loss_D, color='#4f9d69', lw=1.6)
ax[1].axhline(1.0, ls='--', c='#888', lw=1)
ax[1].set_title('Discriminator loss — flat, gives no warning')
ax[1].set_ylim(.85, 1.06)

ax[2].plot(e.index, e.PSNR, color='#2f6f9f', lw=1.5)
ax[2].axvline(COLLAPSE, color='#333', ls='--', lw=1)
ax[2].set_title('Validation PSNR during training'); ax[2].set_ylabel('dB')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

pre, post = e.loc[:COLLAPSE-1], e.loc[COLLAPSE:]
print(f'붕괴 전 최고 PSNR : epoch {pre.PSNR.idxmax():3d}  {pre.PSNR.max():.2f} dB')
print(f'붕괴 후 최고 PSNR : epoch {post.PSNR.idxmax():3d}  {post.PSNR.max():.2f} dB')

In [ ]:
from srgan_models import load_srgan

CK = {'before (ep12)': 'srgan_g_x2_ep12_before.pth',
      'after  (ep84)': 'srgan_g_x2_ep84_after.pth'}
nets = {k: load_srgan(fetch(f'{MODEL}/checkpoints/{v}', v), scale=SCALE) for k, v in CK.items()}

def make(net):
    @torch.no_grad()
    def f(lr):
        t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(net.parameters()).device) / 255
        return (net(t).clamp(0, 1)[0].cpu().numpy().transpose(1, 2, 0) * 255).round().astype('uint8')
    return f

up = {k: make(v) for k, v in nets.items()}
zoom([('Bicubic', bicubic(val_lr, SCALE))] + [(k, up[k](val_lr)) for k in CK] + [('Target', val_tgt)],
     title='validation (Paris), x2')

## 4. 평가

In [ ]:
for k in CK:
    print(f'--- {k} ---')
    compare(up[k], label=f'SRGAN {k}', scale=SCALE, plot=False)
    print()

In [ ]:
rows = compare(up['after  (ep84)'], label='SRGAN after', scale=SCALE)

붕괴 **후** 가중치가 더 좋다.

적대적 항이 상수가 되어 기울기가 사라진 뒤로는 사실상 MSE + VGG 손실만으로 학습되는데,
이 데이터에서는 그쪽이 PSNR·SSIM 에 더 유리했다. GAN 이 항상 이득은 아니라는 뜻이다.

다만 마지막 epoch 100 은 12.01 dB 로 크게 떨어졌다. 붕괴 뒤에도 학습은 불안정하므로
**마지막 가중치가 아니라 best 를 골라 써야 한다.**